In [1]:
import sqlite3

conn = sqlite3.connect("data/local.db")
cursor = conn.cursor()

# DB에 존재하는 모든 테이블 확인
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

# 예시: 특정 테이블의 컬럼 구조 확인
cursor.execute("PRAGMA table_info('session_summary');")
print(cursor.fetchall())

conn.close()


[]
[]


In [2]:
import chromadb

# 기본적으로 ./chroma 폴더에 로컬 DB가 생성되어 있음
client = chromadb.PersistentClient(path="./chroma")

# 저장된 컬렉션 이름 확인
print(client.list_collections())

# 특정 컬렉션 선택
collection = client.get_collection("process_meta")

# 컬렉션에 저장된 문서 확인
docs = collection.get(include=["documents", "metadatas"], limit=5)
print(docs)


[]


ValueError: Collection process_meta does not exist.

In [4]:
import chromadb

client = chromadb.PersistentClient(path="./chroma")
print(client.list_collections())


[]


In [5]:
# rag_query_demo.ipynb

# -----------------------
# 1. 라이브러리 로드
# -----------------------
import os
import json
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# -----------------------
# 2. 환경변수 로드
# -----------------------
load_dotenv()
OPENAI_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_KEY:
    raise RuntimeError("OPENAI_API_KEY가 없습니다. .env 파일 확인하세요.")

# OpenAI LLM 클라이언트
client_oa = OpenAI(api_key=OPENAI_KEY)

# -----------------------
# 3. ChromaDB 연결
# -----------------------
client = chromadb.PersistentClient(path="./data/chroma_openai")

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_KEY,
    model_name="text-embedding-3-small"
)

collection = client.get_or_create_collection(
    name="process_meta",
    embedding_function=openai_ef
)


In [7]:
# -----------------------
# 4. 저장된 메타데이터 일부 확인
# -----------------------
print("=== 현재 process_meta 컬렉션 상태 ===")
print(collection.count(), "개의 문서가 저장됨")

# 5개만 샘플로 가져오기
docs = collection.get(include=["documents", "metadatas"], limit=5)
for i, d in enumerate(docs["documents"]):
    print(f"\n--- Document {i+1} ---")
    print(d[:300], "...")   # 앞부분만 출력
    print("Source:", docs["metadatas"][i]["source"])

=== 현재 process_meta 컬렉션 상태 ===
5 개의 문서가 저장됨

--- Document 1 ---
{"process_id": "LWDED_000224_143655", "process_datetime": "1900-02-24T14:36:55.834000", "file_name": "20250224_Jisu_recorn_cont1_data.csv", "file_path": "C:\\Users\\KAMIC^^\\Desktop\\20250224_Jisu_recorn_cont1_data.csv", "rows": 15161, "cols": 19, "file": "20250224_Jisu_recorn_cont1_data.csv", "colu ...
Source: data\meta_json\LWDED_000224_143655.json

--- Document 2 ---
{"process_id": "LWDED_000224_155257", "process_datetime": "1900-02-24T15:52:57.687000", "file_name": "20250224_Jisu_recorn_cont2_data.csv", "file_path": "C:\\Users\\KAMIC^^\\Desktop\\20250224_Jisu_recorn_cont2_data.csv", "rows": 10795, "cols": 19, "file": "20250224_Jisu_recorn_cont2_data.csv", "colu ...
Source: data\meta_json\LWDED_000224_155257.json

--- Document 3 ---
{"process_id": "LWDED_000224_171102", "process_datetime": "1900-02-24T17:11:02.781000", "file_name": "20250224_Jisu_recorn_cont3_data.csv", "file_path": "C:\\Users\\KAMIC^^\\Desktop\\2025022

In [8]:
# -----------------------
# 5. RAG 질의 테스트
# -----------------------
query = "mpt와 process stability score의 관계를 알려줘"

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas"]
)

print("\n=== RAG 검색 결과 ===")
for i, doc in enumerate(results["documents"][0]):
    print(f"\n[Result {i+1}]")
    print(doc[:500], "...")  # 앞부분만 출력
    print("Source:", results["metadatas"][0][i]["source"])


=== RAG 검색 결과 ===

[Result 1]
{"process_id": "LWDED_000224_180705", "process_datetime": "1900-02-24T18:07:05.995000", "file_name": "20250224_Jisu_recorn_cont5_data.csv", "file_path": "C:\\Users\\KAMIC^^\\Desktop\\20250224_Jisu_recorn_cont5_data.csv", "rows": 499, "cols": 19, "file": "20250224_Jisu_recorn_cont5_data.csv", "columns": {"time": {"dtype": "datetime64[ns]", "non_null": 499, "nulls": 0}, "x": {"dtype": "float64", "non_null": 499, "nulls": 0, "stats": {"min": 0.0, "max": 10.02, "mean": 5.069739478957916}}, "y": {"dt ...
Source: data\meta_json\LWDED_000224_180705.json

[Result 2]
{"process_id": "LWDED_000224_171102", "process_datetime": "1900-02-24T17:11:02.781000", "file_name": "20250224_Jisu_recorn_cont3_data.csv", "file_path": "C:\\Users\\KAMIC^^\\Desktop\\20250224_Jisu_recorn_cont3_data.csv", "rows": 1750, "cols": 19, "file": "20250224_Jisu_recorn_cont3_data.csv", "columns": {"time": {"dtype": "datetime64[ns]", "non_null": 1750, "nulls": 0}, "x": {"dtype": "float64", "non_

In [9]:
# -----------------------
# 6. LLM에 context 포함해 질의
# -----------------------
context = "\n\n".join(results["documents"][0])

prompt = f"""
당신은 공정 데이터 전문가입니다.
다음 메타데이터를 참고하여 질문에 답하세요.

[Context]
{context}

[Question]
{query}
"""

response = client_oa.chat.completions.create(
    model="gpt-4o-mini",  # 필요 시 gpt-4o로 변경 가능
    messages=[{"role": "system", "content": "당신은 데이터 분석 전문가입니다."},
              {"role": "user", "content": prompt}]
)

print("\n=== LLM 응답 ===")
print(response.choices[0].message.content)


=== LLM 응답 ===
mpt(Mean Process Time)와 process stability score는 공정 데이터 분석에서 중요한 두 지표로, 각각의 정의와 그 관계는 다음과 같습니다.

1. **MPT (Mean Process Time)**:
   - MPT는 특정 공정에서 각 작업의 평균 소요 시간을 측정한 것입니다. 이 값은 공정의 효율성과 생산성을 나타내며, 일반적으로 MPT가 짧을수록 공정이 더 효율적이라는 의미를 가집니다.
   - 제공된 메타데이터에서는 MPT의 평균 값, 최대 값, 표준 편차 등이 각각의 공정 프로세스에 대해 다르게 나타납니다.

2. **Process Stability Score**:
   - Process stability score는 공정의 안정성을 평가하는 지표입니다. 이 점수는 온도 안정성, 하중 안정성 등 여러 요소를 바탕으로 계산됩니다. 높은 안정성 점수는 공정이 일관되게 실행되고 있음을 의미합니다.

### 관계
- **효율성과 안정성의 상관관계**: 
  - 이상적으로는 MPT가 짧으면서도 stability score가 높은 공정이 가장 바람직합니다. 이는 공정이 효율적일 뿐만 아니라, 일관되게 제어되고 있음을 나타냅니다.
  - 그러나 안정성이 떨어지는 공정에서는 예기치 않은 변동이 자주 발생 할 수 있으며, 이로 인해 평균 소요 시간이 증가할 가능성이 있습니다. 따라서 MPT가 긴 경우라도 그 이유가 불안정성 때문이라면, 이는 개선의 여지가 있음을 나타냅니다.

- **메타데이터 분석 예시**:
  - 예를 들어, 세 개의 공정 데이터에서 MPT가 각각 다르게 나온다고 가정했을 때, MPT와 stability score를 비교하여 어떤 공정이 가장 효율적이며 안정적인지 판단할 수 있습니다. 
  - 데이터를 분석한 결과, MPT가 짧은 공정이 가장 높은 stability score를 가질 경우, 이는 효율성과 안정성이 동시에 성취된 상태임을 나타냅니다.

### 결론
결론적으로, MPT와 proc

In [10]:
import json
from openai import OpenAI

# OpenAI 클라이언트 초기화 (환경변수에 OPENAI_API_KEY 저장 필요)
client = OpenAI()

# RAG 검색 결과 예시 (앞에서 collection.query로 얻은 results 활용)
query = "mpt와 process stability score의 관계를 알려줘"
results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas"]
)

# 검색된 JSON 문서들
docs = results["documents"][0]

# LLM에 context로 전달
context = "\n\n".join(docs)

prompt = f"""
당신은 제조 공정 데이터 분석 전문가입니다.
다음은 데이터 파일에서 추출한 메타데이터(JSON)입니다:

{context}

위 정보를 참고하여, 질문에 대답하세요:

질문: {query}

답변은 사람이 이해할 수 있도록 한국어로 요약 설명해 주세요.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",  # 또는 gpt-4o, gpt-4.1 등
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)

print("=== 최종 요약 답변 ===")
print(response.choices[0].message.content)


=== 최종 요약 답변 ===
mpt(Mean Process Time)는 제조 공정에서 특정 작업을 수행하는 데 걸리는 평균 시간을 나타내며, 공정의 효율성과 성능을 평가하는 중요한 지표입니다. 반면, process stability score는 공정의 안정성을 평가하는 지표로, 온도 안정성, 하중 안정성 등 다양한 요소를 종합하여 최종 점수를 제공합니다.

mpt와 process stability score 간의 관계를 살펴보면, 일반적으로 mpt가 낮을수록 공정이 더 효율적이고 안정적일 가능성이 높습니다. 즉, 공정이 안정적일수록 작업이 원활하게 진행되어 평균 처리 시간이 짧아질 수 있습니다. 반대로, mpt가 높거나 변동성이 클 경우, 이는 공정의 불안정성을 나타낼 수 있으며, 이로 인해 process stability score가 낮아질 수 있습니다.

예를 들어, 제공된 데이터에서 첫 번째 공정(LWDED_000224_180705)의 mpt는 약 706.41초이고, process stability score는 71.4로 나타났습니다. 두 번째 공정(LWDED_000224_171102)의 mpt는 약 707.22초로 비슷하지만, stability score는 70.9로 약간 낮습니다. 이는 두 공정의 mpt가 비슷하지만, 안정성 측면에서 약간의 차이가 있음을 보여줍니다.

결론적으로, mpt와 process stability score는 서로 연관되어 있으며, 공정의 효율성과 안정성을 평가하는 데 중요한 역할을 합니다. 안정적인 공정은 일반적으로 더 낮은 mpt를 기록할 가능성이 높습니다.


In [11]:
import json
from openai import OpenAI

client = OpenAI()

# RAG 검색 결과
query = "mpt와 process stability score의 관계를 알려줘"
results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas"]
)

docs = results["documents"][0]

context = "\n\n".join(docs)

# 도메인 사전 정의를 프롬프트에 추가
domain_knowledge = """
용어 정의:
- MPT: Melt Pool Temperature, 용융풀의 온도를 의미하며 공정 안정성과 직결되는 핵심 지표입니다.
- Process Stability Score: 공정 중 온도 및 부하의 변동성을 종합적으로 반영한 안정성 점수입니다.
"""

prompt = f"""
당신은 제조 공정 데이터 분석 전문가입니다.

{domain_knowledge}

다음은 데이터 파일에서 추출한 메타데이터(JSON)입니다:

{context}

위 정보를 참고하여, 질문에 대답하세요:

질문: {query}

답변은 사람이 이해할 수 있도록 한국어로 요약 설명해 주세요.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)

print("=== 최종 요약 답변 ===")
print(response.choices[0].message.content)


=== 최종 요약 답변 ===
MPT(용융풀 온도)와 Process Stability Score(공정 안정성 점수) 간의 관계는 제조 공정의 품질과 안정성을 평가하는 데 중요한 역할을 합니다.

1. **MPT의 중요성**: MPT는 공정의 온도를 나타내며, 이는 용융풀의 안정성과 직접적으로 연결됩니다. MPT가 일정하게 유지되면, 공정의 품질이 높아지고 결함이 줄어듭니다.

2. **Process Stability Score**: 이 점수는 온도와 부하의 변동성을 종합적으로 반영하여 공정의 안정성을 평가합니다. 높은 점수는 공정이 안정적이라는 것을 의미하며, 이는 MPT가 일정하게 유지되고 있다는 것을 나타냅니다.

3. **상관관계**: 일반적으로 MPT가 안정적으로 유지될수록 Process Stability Score도 높아집니다. 즉, MPT의 변동성이 적을수록 공정의 안정성이 높아지고, 이는 최종적으로 품질 향상으로 이어집니다.

결론적으로, MPT와 Process Stability Score는 서로 밀접하게 연결되어 있으며, MPT의 안정성이 공정의 전반적인 안정성과 품질에 긍정적인 영향을 미친다고 할 수 있습니다.


In [13]:
import json
import chromadb
from chromadb.utils import embedding_functions

# OpenAI 임베딩 함수 정의
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key="YOUR_OPENAI_API_KEY",
    model_name="text-embedding-3-small"
)

# ChromaDB PersistentClient 생성
client = chromadb.PersistentClient(path="./chroma_data")

# 컬렉션 생성/가져오기
collection = client.get_or_create_collection(
    name="lw_ded_columns",
    embedding_function=openai_ef
)

# JSON 파일 로드 (data/meta_json 경로로 수정됨)
with open("data/meta_json/lw_ded_columns.json", "r", encoding="utf-8") as f:
    columns = json.load(f)

# 업로드 준비
documents = [f"{c['column']} ({c['name']}): {c['description']}" for c in columns]
metadatas = columns
ids = [f"col_{c['column']}" for c in columns]

# ChromaDB에 업로드
collection.upsert(documents=documents, metadatas=metadatas, ids=ids)

print("✅ LW-DED 컬럼 정의가 ChromaDB에 저장되었습니다.")


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [14]:
with open("data/meta_json/lw_ded_columns.json", "r", encoding="utf-8") as f:
    raw = f.read()
    print("파일 내용 미리보기:", raw[:200])  # 앞부분만 출력


파일 내용 미리보기: [
  {
    "column": "process_id",
    "name": "공정 식별자",
    "description": "해당 LW-DED 데이터의 고유 번호로, 추적 및 관리에 사용."
  },
  {
    "column": "process_datetime",
    "name": "공정 시각",
    "description": "공정 
